# Inference networks: the same epidemic, three posterior estimators

Compare Coupling Flow, Flow Matching, and Diffusion Model using one SIR observation model.
This notebook runs independently of the summary-network notebook.

Run from this directory with `requirements.txt` (BayesFlow 2.1.0). JAX is the default backend.
Flow matching and diffusion require numerical integration during sampling.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

import time
import keras
import numpy as np
import matplotlib.pyplot as plt
import bayesflow as bf

from scipy.special import expit, logit, logsumexp
from scipy.stats import binom

SEED = 2026
keras.utils.set_random_seed(SEED)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
print(f"BayesFlow {bf.__version__}; Keras backend: {keras.backend.backend()}")

## An epidemic observed through surveys

SIR tracks susceptible, infected, and recovered fractions of a population:

$$\frac{dS}{dt}=-\beta SI,\qquad
\frac{dI}{dt}=\beta SI-\gamma I,\qquad
\frac{dR}{dt}=\gamma I.$$

Time is measured in days. Transmission $\beta$ and recovery $\gamma$ have independent
uniform priors, $\beta\sim U(0.15,0.65)$ and $\gamma\sim U(0.05,0.25)$.
The initial state is known: $(S_0,I_0,R_0)=(0.99,0.01,0)$.

Every two days, a fresh survey samples 200 people. Conditional on the deterministic
trajectory, survey counts are independent:

$$y_j\mid\beta,\gamma\sim\operatorname{Binomial}(200,I(t_j;\beta,\gamma)).$$

The randomness here comes from observation, rather than stochastic disease transmission.
This is a teaching model: no changing behavior, underreporting, or uncertain initial state.
We implement the compartment flows, numerical solution, and survey observations below.

In [ ]:
TIMES = np.arange(2.0, 61.0, 2.0)  # Survey dates, in days.
SURVEY_SIZE = 200
INITIAL_STATE = np.array([0.99, 0.01, 0.0])  # Susceptible, infected, recovered fractions.
LOWER = np.array([0.15, 0.05])  # Lower prior bounds for beta and gamma.
UPPER = np.array([0.65, 0.25])
PARAMETER_NAMES = (r"$\beta$", r"$\gamma$")

### Movement between compartments

`new_infections` moves people from S to I; `new_recoveries` moves people from I to R.
Each departure is an arrival elsewhere, so the three derivatives sum to zero and
$S+I+R=1$ throughout the outbreak. Recovery gives lasting immunity in this model.

The function below evaluates these flows for a batch of outbreaks. `state` has one
row per outbreak and three columns: susceptible, infected, recovered.

In [ ]:
def sir_derivative(state, beta, gamma):
    """Rates of change of the susceptible, infected, and recovered fractions."""
    susceptible, infected, recovered = state.T
    new_infections = beta * susceptible * infected
    new_recoveries = gamma * infected
    return np.column_stack((
        -new_infections,                    # Susceptible people become infected.
        new_infections - new_recoveries,    # Infected people enter and leave this group.
        new_recoveries,                     # Recoveries accumulate.
    ))

### Follow the epidemic through time

`trajectories` starts every outbreak from `INITIAL_STATE` and integrates the equations
up to each survey date. We use fourth-order Runge–Kutta (RK4) with steps of at most
a quarter day. The four slopes `k1` through `k4` are numerical integration steps;
the epidemic model itself is entirely in `sir_derivative`.

Outbreaks are solved together using NumPy arrays so generating training data stays fast.
The returned shape is `(number of outbreaks, number of dates, 3)`.

In [ ]:
def trajectories(rates, times=TIMES, step=0.25):
    """Vectorized RK4 solution; return (batch, time, S/I/R compartment)."""
    rates = np.atleast_2d(np.asarray(rates, dtype=float))
    times = np.asarray(times, dtype=float)
    if rates.shape[1] != 2 or np.any(rates <= 0) or not np.isfinite(rates).all():
        raise ValueError("rates must contain positive, finite (beta, gamma) pairs")
    if times.ndim != 1 or len(times) == 0 or times[0] < 0 or np.any(np.diff(times) <= 0):
        raise ValueError("times must be nonnegative and strictly increasing")
    if step <= 0:
        raise ValueError("step must be positive")
    beta, gamma = rates.T
    state = np.broadcast_to(INITIAL_STATE, (len(rates), 3)).copy()

    # Rows are independent outbreaks; columns are S, I, and R.
    result = np.empty((len(rates), len(times), 3))
    current = 0.0
    for index, target in enumerate(times):
        n_steps = max(1, int(np.ceil((target - current) / step)))
        h = (target - current) / n_steps
        # RK4 combines four slopes to advance all outbreaks by h days.
        for _ in range(n_steps):
            k1 = sir_derivative(state, beta, gamma)
            k2 = sir_derivative(state + h * k1 / 2, beta, gamma)
            k3 = sir_derivative(state + h * k2 / 2, beta, gamma)
            k4 = sir_derivative(state + h * k3, beta, gamma)
            state += h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        result[:, index] = state
        current = target
    return result

In [ ]:
example = trajectories([[0.38, 0.12]])[0]
fig, ax = plt.subplots(figsize=(8, 3.5))
for column, label in enumerate(["Susceptible", "Infected", "Recovered"]):
    ax.plot(TIMES, example[:, column], label=label)
ax.set(xlabel="Day", ylabel="Population fraction")
ax.legend()
plt.show()
print("Largest deviation from S + I + R = 1:", np.max(np.abs(example.sum(axis=1) - 1)))

### Observe the epidemic

The simulator first draws transmission and recovery rates from their priors, then solves
the corresponding epidemic. Each survey reports an infected count out of 200 people.
A fresh independent sample is taken at every date; these are not repeated observations
of the same 200 people.

`series` stores observed fractions in date order. `surveys` pairs each fraction with its
date, scaled to the interval [0, 1]. `counts` retains the integer observations for the
reference likelihood. The target-coordinate mapping is defined in the next section.

In [ ]:
def simulate(n, rng, rates=None):
    """Draw outbreaks and independent surveys on a fixed two-day schedule."""
    if rates is None:
        rates = rng.uniform(LOWER, UPPER, size=(n, 2))
    else:
        rates = np.atleast_2d(rates)
        if len(rates) != n:
            raise ValueError("n must match the number of supplied rate pairs")
    # First solve the latent epidemic, then sample what a survey would observe.
    prevalence = trajectories(rates)[:, :, 1]
    counts = rng.binomial(SURVEY_SIZE, prevalence)
    # Two representations of the same survey counts for the summary networks.
    series = (counts / SURVEY_SIZE).astype("float32")[..., None]
    dates = np.broadcast_to(TIMES / TIMES[-1], series.shape[:2])
    return {
        "parameters": to_unconstrained(rates).astype("float32"),
        "surveys": np.stack((dates, series[..., 0]), axis=-1).astype("float32"),
        "series": series,
        "counts": counts,
    }

In [ ]:
examples = np.array([[0.25, 0.15], [0.45, 0.15], [0.45, 0.08]])
curves = trajectories(examples)
fig, ax = plt.subplots(figsize=(8, 3.5))
for rates, curve in zip(examples, curves):
    ax.plot(TIMES, curve[:, 1], label=rf"$\beta={rates[0]:.2f},\ \gamma={rates[1]:.2f}$")
ax.set(xlabel="Day", ylabel="Infected fraction")
ax.legend()
plt.show()

## Targets and training budget

Both notebooks use the same prior, surveys, and train/validation/test split.
Each dataset is one outbreak with 30 surveys; the test outbreaks are drawn independently.

Networks infer unconstrained coordinates
$u_k=\operatorname{logit}((\theta_k-a_k)/(b_k-a_k))$.
`to_rates` reverses this mapping, so every posterior draw respects the prior bounds.
BayesFlow standardizes targets and observation channels using training-data statistics.
This keeps the prevalence signal on a useful scale alongside the time embedding.
All diagnostics below use rates
in their original units.

The settings below are a starting budget, not a convergence guarantee. Inspect validation
loss and calibration before drawing conclusions. For an initial execution check, reduce
the dataset sizes and epochs; that run cannot assess posterior quality.

In [ ]:
def to_unconstrained(rates):
    """Map rates inside the uniform prior bounds to the real line."""
    return logit((np.asarray(rates) - LOWER) / (UPPER - LOWER))


def to_rates(parameters):
    """Map network outputs back to transmission and recovery rates (per day)."""
    return LOWER + (UPPER - LOWER) * expit(parameters)

In [ ]:
N_TRAIN = 4096
N_VALIDATION = 512
N_TEST = 128
EPOCHS = 20
BATCH_SIZE = 128
N_DRAWS = 256
N_OBSERVED_DRAWS = 1024
GRID_POINTS = 160

# Separate random streams keep the observed outbreak independent of training size.
streams = np.random.SeedSequence(SEED).spawn(5)
train = simulate(N_TRAIN, np.random.default_rng(streams[0]))
validation = simulate(N_VALIDATION, np.random.default_rng(streams[1]))
test = simulate(N_TEST, np.random.default_rng(streams[2]))
true_rates = np.array([[0.38, 0.12]])
observed = simulate(1, np.random.default_rng(streams[3]), rates=true_rates)
rng = np.random.default_rng(streams[4])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(TIMES, trajectories(true_rates)[0, :, 1], label="Underlying prevalence")
ax.scatter(TIMES, observed["series"][0, :, 0], label="Survey", s=20)
ax.set(xlabel="Day", ylabel="Infected fraction")
ax.legend()
plt.show()

## Three ways to learn a posterior

| Network | Training task | Posterior sampling |
|---|---|---|
| Coupling Flow | Conditional density estimation with invertible transformations | Transform a base draw through the coupling layers |
| Flow Matching | Learn a velocity field along paths between parameters and base noise | Integrate the learned ODE |
| Diffusion Model | Learn denoising across noise levels | Integrate the reverse process; here we use its probability-flow ODE |

Flow Matching uses its default adaptive ODE integrator. For Diffusion Model we explicitly
select the adaptive `tsit5` ODE integrator instead of its default stochastic sampler. Each estimator gets a fresh
Time Series Transformer with the same architecture, the same training examples, and the
same number of optimizer updates. This compares complete estimators; the learned summaries
are not shared. Parameter counts and training costs differ.

Loss scales differ across objectives. Compare posterior diagnostics, rather than the absolute
loss of one method against another. Equal epochs do not imply equal convergence.

In [ ]:
inference_networks = {
    "Coupling Flow": bf.networks.CouplingFlow(
        depth=4, subnet_kwargs={"widths": (64, 64), "dropout": 0.0},
    ),
    "Flow Matching": bf.networks.FlowMatching(
        subnet_kwargs={"widths": (64, 64, 64)},
    ),
    "Diffusion Model": bf.networks.DiffusionModel(
        subnet_kwargs={"widths": (64, 64, 64)},
        integrate_kwargs={"method": "tsit5"},
    ),
}
workflows, histories, training_seconds = {}, {}, {}
for name, inference_network in inference_networks.items():
    workflow = bf.BasicWorkflow(
        summary_network=bf.networks.TimeSeriesTransformer(
            summary_dim=8, embed_dims=(32, 32), num_heads=(2, 2), dropout=0.0,
        ),
        inference_network=inference_network,
        inference_variables=["parameters"], summary_variables=["series"],
        standardize=["inference_variables", "summary_variables"],
    )
    started = time.perf_counter()
    histories[name] = workflow.fit_offline(
        train, validation_data=validation, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
    )
    training_seconds[name] = time.perf_counter() - started
    workflows[name] = workflow
    print(f"{name}: {training_seconds[name]:.1f} seconds, including compilation")

In [ ]:
for name, history in histories.items():
    fig = bf.diagnostics.loss(history, smoothing_factor=0)
    fig.suptitle(name, y=1.03)
    plt.show()

## Recovery and calibration on new outbreaks

BayesFlow's recovery plot compares posterior medians and uncertainty with the generating
rates. SBC rank ECDFs and coverage curves check uncertainty across independent prior-predictive
test outbreaks. A good fit to one outbreak is insufficient evidence of calibration.
With 128 test outbreaks, these checks are preliminary; use more for a final assessment.

In [ ]:
test_draws = {}
targets = to_rates(test["parameters"])
for index, (name, workflow) in enumerate(workflows.items()):
    draws = workflow.sample(
        num_samples=N_DRAWS, conditions=test, batch_size=32, seed=SEED + 100 + index,
    )["parameters"]
    test_draws[name] = to_rates(draws)
    # Plot recovery
    # Your code here
    fig.suptitle(name, y=1.03)
    plt.show()

In [ ]:
for name, draws in test_draws.items():
    # Plot ECDF
    # Your code here
    fig.suptitle(name, y=1.03)
    plt.show()
    # Plot coverage
    # Your code here
    fig.suptitle(name, y=1.03)
    plt.show()

In [ ]:
for name, draws in test_draws.items():
    print(name)
    print("  RMSE:", bf.diagnostics.root_mean_squared_error(
        estimates=draws, targets=targets, variable_names=PARAMETER_NAMES, normalize=None,
    ))
    print("  Calibration error:", bf.diagnostics.calibration_error(
        estimates=draws, targets=targets, variable_names=PARAMETER_NAMES,
    ))